# PhoBERT Improved Training
**Branch:** `feature/improve-training`  
**Cải tiến so với baseline:**
1. Early stopping dùng **Combined F1** (thay vì dev_loss)
2. **Learning rate = 2e-5** (thay vì 1e-4)
3. **concat_4_layers** encoder (thay vì cls_only)
4. **Ensemble top-3 checkpoint** tự động sau training


In [ ]:
import torch

if torch.cuda.is_available():
    gpu  = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU: {gpu} | VRAM: {vram:.1f} GB')
    if vram < 14:
        print('  [WARN] VRAM < 14GB — batch_size có thể cần giảm')
else:
    raise RuntimeError('Không có GPU!')

torch.cuda.empty_cache()
print(f'PyTorch: {torch.__version__} | CUDA: {torch.version.cuda}')

import subprocess
subprocess.run([
    'pip', 'install', '-q',
    'transformers>=4.41.0', 'underthesea', 'py_vncorenlp',
    'tabulate', 'tqdm', 'scikit-learn', 'sentencepiece', 'sacremoses'
], check=True)
print('Dependencies installed')

In [ ]:
import os, sys

try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('GITHUB_TOKEN')
    REPO_URL = f'https://{token}@github.com/vudinhminh08/NLP-project-master-study.git'
    print('GitHub token loaded from Kaggle Secrets')
except Exception as e:
    print(f'[WARN] Không lấy được token: {e}')
    REPO_URL = 'https://github.com/vudinhminh08/NLP-project-master-study.git'

REPO_BRANCH = 'feature/improve-training'
PROJECT_DIR = '/kaggle/working/absa-project'

if not os.path.exists(PROJECT_DIR):
    import subprocess
    subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, '--depth=1',
                    REPO_URL, PROJECT_DIR], check=True)
    print('Clone completed')
else:
    import subprocess
    subprocess.run(['git', '-C', PROJECT_DIR, 'pull', 'origin', REPO_BRANCH], check=True)
    print('Repo pulled')

os.chdir(PROJECT_DIR)
print(f'Working dir: {os.getcwd()}')

for d in ['data', 'outputs/models_lr2e5', 'outputs/results_lr2e5', 'outputs/eda']:
    os.makedirs(d, exist_ok=True)

sys.path.insert(0, 'code/data_processing')
sys.path.insert(0, 'code/phobert')

import subprocess
result = subprocess.run(['ls', 'code/phobert/'], capture_output=True, text=True)
print(result.stdout)

In [ ]:
import pandas as pd, os

files_needed = ['data/train.csv', 'data/dev.csv', 'data/test.csv']
all_exist = all(os.path.exists(f) for f in files_needed)

if all_exist:
    print('Data files already exist')
else:
    try:
        from datasets import load_dataset
        ds = load_dataset('ds4v/absa-vlsp-2018', 'hotel')
        for split in ['train', 'dev', 'test']:
            ds[split].to_pandas().to_csv(f'data/{split}.csv', index=False)
            print(f'{split}: {len(ds[split])} samples saved')
    except Exception as e:
        print(f'[ERROR] {e}')
        raise

for f in files_needed:
    df = pd.read_csv(f)
    print(f'{f}: {len(df)} rows')

In [ ]:
import os

# Preprocessing (baseline VnCoreNLP, không error correction)
preprocessed_ok = all(
    os.path.exists(f'data/{s}_preprocessed.csv')
    for s in ['train', 'dev', 'test']
)

if preprocessed_ok:
    print('Preprocessed data already exists')
else:
    print('Running preprocessing...')
    import subprocess
    result = subprocess.run(
        ['python', 'code/data_processing/step3_preprocessing.py'],
        capture_output=True, text=True
    )
    print(result.stdout[-2000:])
    if result.returncode != 0:
        print('STDERR:', result.stderr[-1000:])

# EDA artifacts (class_weights.json, encoder_config.json)
eda_ok = (
    os.path.exists('outputs/eda/class_weights.json') and
    os.path.exists('outputs/eda/encoder_config.json')
)
if not eda_ok:
    print('Running EDA...')
    import subprocess
    subprocess.run(['python', 'code/data_processing/step2_eda.py'], check=True)

# Đặt encoder_config = concat_4_layers
import json
enc_cfg_path = 'outputs/eda/encoder_config.json'
with open(enc_cfg_path, 'w') as f:
    json.dump({'encoder_option': 'concat_4_layers'}, f)
print('encoder_config.json set to concat_4_layers')

In [ ]:
import json, os
from utils.constants import TRAIN_CONFIG, ZERO_TRAIN_ASPECTS, PHOBERT_MODEL_NAME

enc_cfg = json.load(open('outputs/eda/encoder_config.json'))
print('=== Encoder Config ===')
for k, v in enc_cfg.items():
    print(f'  {k}: {v}')

cw = json.load(open('outputs/eda/class_weights.json'))
print('\n=== Global Class Weights ===')
label_map = {'0': 'absent', '1': 'positive', '2': 'negative', '3': 'neutral'}
for cls, w in cw['global_weights'].items():
    print(f"  {label_map.get(cls,cls):12s}: {float(w):.1f}x")

print('\n=== Train Config (base) ===')
for k, v in TRAIN_CONFIG.items():
    print(f'  {k}: {v}')
print(f'  model: {PHOBERT_MODEL_NAME}')
print(f'\n  → lr override: 2e-5 (truyền trực tiếp vào run_main)')
print(f'  → early stop criterion: Combined F1 (không phải dev_loss)')
print(f'  ZERO_TRAIN_ASPECTS: {ZERO_TRAIN_ASPECTS}')
print('\nConfig OK')


In [ ]:
# Train: concat_4_layers + lr=2e-5 + Combined F1 criterion + ensemble top-3
# Output: outputs/models_lr2e5/ và outputs/results_lr2e5/
# (không ghi đè kết quả baseline cũ)

from run_experiment import main as run_main

test_metrics = run_main(
    encoder_option='concat_4_layers',
    use_amp=True,
    lr=2e-5,
    data_suffix='',
)
print('\nTest metrics:', test_metrics)

In [ ]:
import json, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

history = json.load(open('outputs/results_lr2e5/training_history.json'))
best_ep = history['best_epoch']
epochs  = range(1, len(history['train_loss']) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('PhoBERT concat_4_layers lr=2e-5 — Learning Curve (ABSA VLSP 2018)', fontsize=13)

ax1.plot(epochs, history['train_loss'], 'o-', c='crimson',   lw=2, label='Train Loss')
ax1.plot(epochs, history['dev_loss'],   'o-', c='steelblue', lw=2, label='Dev Loss')
ax1.axvline(best_ep, c='green', ls='--', alpha=0.7, label=f'Best (epoch {best_ep})')
ax1.set(title='Loss', xlabel='Epoch', ylabel='Cross-Entropy Loss')
ax1.legend(); ax1.grid(alpha=0.3)

ax2.plot(epochs, history['dev_acd_f1'],      's-', c='darkorange', lw=2,   label='Dev ACD F1')
ax2.plot(epochs, history['dev_spc_f1'],      '^-', c='purple',     lw=2,   label='Dev SPC F1')
ax2.plot(epochs, history['dev_combined_f1'], 'o-', c='green',      lw=2.5, label='Dev Combined F1')
ax2.axvline(best_ep, c='green', ls='--', alpha=0.7, label=f'Best (epoch {best_ep})')
ax2.axhline(0.7732,  c='red',   ls=':',  alpha=0.5, label='SOTA Combined 0.7732')
ax2.set(title='F1 Score', xlabel='Epoch', ylabel='Macro F1')
ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('outputs/eda/learning_curve_lr2e5.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nPhân tích learning curve:')
print(f'  Best epoch       : {best_ep} / {len(history["train_loss"])}')
print(f'  Best Combined F1 : {history["best_combined_f1"]:.4f}')
if len(history['train_loss']) > best_ep:
    print(f'  Train loss @ {best_ep}: {history["train_loss"][best_ep-1]:.4f} → {history["train_loss"][best_ep]:.4f}')
    print(f'  Dev loss   @ {best_ep}: {history["dev_loss"][best_ep-1]:.4f} → {history["dev_loss"][best_ep]:.4f}')
print(f'  Saved: outputs/eda/learning_curve_lr2e5.png')


In [ ]:
import json, os
from tabulate import tabulate

rows = []

# Baseline (concat_4_layers, lr=1e-4) — output dir không có suffix
baseline_path = 'outputs/results/phobert_test_metrics.json'
if os.path.exists(baseline_path):
    m = json.load(open(baseline_path))
    rows.append(['Baseline (concat_4_layers, lr=1e-4)',
                 m['macro_acd_f1'], m['macro_spc_f1'], m['macro_combined_f1']])

# Improved (concat_4_layers, lr=2e-5)
improved_path = 'outputs/results_lr2e5/phobert_test_metrics.json'
if os.path.exists(improved_path):
    m = json.load(open(improved_path))
    rows.append(['Improved (concat_4_layers, lr=2e-5, Combined F1 criterion)',
                 m['macro_acd_f1'], m['macro_spc_f1'], m['macro_combined_f1']])

# Ensemble
ens_path = 'outputs/results_lr2e5/phobert_ensemble_test_metrics.json'
if os.path.exists(ens_path):
    m = json.load(open(ens_path))
    rows.append(['Ensemble top-3 (concat_4_layers, lr=2e-5)',
                 m['macro_acd_f1'], m['macro_spc_f1'], m['macro_combined_f1']])

rows.append(['SOTA (ds4v 2022)', 0.8255, '-', 0.7732])

print(tabulate(rows,
    headers=['Method', 'ACD F1', 'SPC F1', 'Combined F1'],
    tablefmt='github', floatfmt='.4f'))

In [ ]:
import os, json, shutil

# In summary report
report = 'outputs/results_lr2e5/phobert_summary.md'
if os.path.exists(report):
    print(open(report, encoding='utf-8').read())
else:
    print('Chưa có report')

# Liệt kê tất cả files kết quả
print('\n=== Kết quả đã tạo ===')
for root, dirs, files in os.walk('outputs'):
    for f in sorted(files):
        if not f.endswith('.DS_Store'):
            path = os.path.join(root, f)
            print(f'  {path} ({os.path.getsize(path)/1024:.1f} KB)')

# Tạo zip toàn bộ outputs để download
zip_path = '/kaggle/working/phobert_improved_results.zip'
shutil.make_archive(
    zip_path.replace('.zip', ''),
    'zip',
    root_dir='/kaggle/working/absa-project',
    base_dir='outputs/results_lr2e5'
)
print(f'\nZip: {zip_path} ({os.path.getsize(zip_path)/1024:.1f} KB)')
print('→ Kaggle Output panel → Download')

# Tổng hợp nhanh
print('\n=== Tổng hợp kết quả ===')
for label, path in [
    ('Baseline  (lr=1e-4)', 'outputs/results/phobert_test_metrics.json'),
    ('Improved  (lr=2e-5)', 'outputs/results_lr2e5/phobert_test_metrics.json'),
    ('Ensemble top-3     ', 'outputs/results_lr2e5/phobert_ensemble_test_metrics.json'),
]:
    if os.path.exists(path):
        m = json.load(open(path))
        print(f'  {label}: ACD={m["macro_acd_f1"]:.4f}  SPC={m["macro_spc_f1"]:.4f}  Combined={m["macro_combined_f1"]:.4f}')
print(f'  SOTA (ds4v 2022)   : ACD=0.8255  Combined=0.7732')
